# Apêndice 10.1: Encadeamento de Prompts

- [Lição](#lesson)
- [Playground de Exemplos](#example-playground)

## Configuração

Execute a célula de configuração abaixo para carregar sua chave de API e estabelecer a função auxiliar `get_completion`.

In [ ]:
import sys, re
sys.path.insert(0, '/workspaces/prompt-eng-ia')
from ollama_helper import get_completion, MODEL_NAME, client

---

## Lição

O ditado diz que "Escrever é reescrever." Acontece que **o Claude frequentemente pode melhorar a precisão de sua resposta quando solicitado a fazê-lo**!

Há muitas maneiras de fazer o Claude "pensar novamente". As formas que parecem naturais para pedir a um humano que verifique seu trabalho geralmente também funcionarão para o Claude. (Confira nossa [documentação de encadeamento de prompts](https://docs.anthropic.com/claude/docs/chain-prompts) para mais exemplos de quando e como usar o encadeamento de prompts.)

### Exemplos

Neste exemplo, pedimos ao Claude para criar dez palavras... mas uma ou mais delas não é uma palavra real.

In [ ]:
# Prompt inicial
first_user = "Name ten words that all end with the exact letters 'ab'."

# Array de mensagens da API
messages = [
    {
        "role": "user",
        "content": first_user
    }
]

# Armazena e imprime a resposta do Claude
first_response = get_completion(messages)
print(first_response)

**Pedir ao Claude para tornar sua resposta mais precisa** corrige o erro! 

Abaixo, recuperamos a resposta incorreta do Claude acima e adicionamos mais um turno à conversa pedindo ao Claude para corrigir sua resposta anterior.

In [ ]:
second_user = "Please find replacements for all 'words' that are not real words."

# Array de mensagens da API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Imprime a resposta do Claude
print("------------------------ Array completo de mensagens com substituições de variáveis ------------------------")
print(messages)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(messages))

Mas o Claude está revisando sua resposta apenas porque dissemos a ele? E se já começarmos com uma resposta correta? O Claude perderá sua confiança? Aqui, colocamos uma resposta correta no lugar de `first_response` e pedimos para verificar novamente.

In [ ]:
first_user = "Name ten words that all end with the exact letters 'ab'."

first_response = """Here are 10 words that end with the letters 'ab':

1. Cab
2. Dab
3. Grab
4. Gab
5. Jab
6. Lab
7. Nab
8. Slab
9. Tab
10. Blab"""

second_user = "Please find replacements for all 'words' that are not real words."

# Array de mensagens da API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Imprime a resposta do Claude
print("------------------------ Array completo de mensagens com substituições de variáveis ------------------------")
print(messages)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(messages))

Você pode notar que se gerar uma resposta do bloco acima algumas vezes, o Claude deixa as palavras como estão na maioria das vezes, mas ainda ocasionalmente muda as palavras mesmo que todas já estejam corretas. O que podemos fazer para mitigar isso? Conforme o Capítulo 8, podemos dar ao Claude uma saída! Vamos tentar mais uma vez.

In [ ]:
first_user = "Name ten words that all end with the exact letters 'ab'."

first_response = """Here are 10 words that end with the letters 'ab':

1. Cab
2. Dab
3. Grab
4. Gab
5. Jab
6. Lab
7. Nab
8. Slab
9. Tab
10. Blab"""

second_user = "Please find replacements for all 'words' that are not real words. If all the words are real words, return the original list."

# Array de mensagens da API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Imprime a resposta do Claude
print("------------------------ Array completo de mensagens com substituições de variáveis ------------------------")
print(messages)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(messages))

Tente gerar respostas do código acima algumas vezes para ver que o Claude é muito melhor em manter sua posição agora.

Você também pode usar o encadeamento de prompts para **pedir ao Claude para melhorar suas respostas**. Abaixo, pedimos ao Claude para primeiro escrever uma história, e depois melhorar a história que escreveu. Seu gosto pessoal pode variar, mas muitos podem concordar que a segunda versão do Claude é melhor.

Primeiro, vamos gerar a primeira versão da história do Claude.

In [ ]:
# Prompt inicial
first_user = "Write a three-sentence short story about a girl who likes to run."

# Array de mensagens da API
messages = [
    {
        "role": "user",
        "content": first_user
    }
]

# Armazena e imprime a resposta do Claude
first_response = get_completion(messages)
print(first_response)

Agora vamos fazer o Claude melhorar seu primeiro rascunho.

In [ ]:
second_user = "Make the story better."

# Array de mensagens da API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Imprime a resposta do Claude
print("------------------------ Array completo de mensagens com substituições de variáveis ------------------------")
print(messages)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(messages))

Esta forma de substituição é muito poderosa. Temos usado placeholders de substituição para passar listas, palavras, respostas anteriores do Claude e assim por diante. Você também pode **usar substituição para fazer o que chamamos de "chamada de função", que é pedir ao Claude para realizar alguma função, e depois pegar os resultados dessa função e pedir ao Claude para fazer ainda mais com os resultados**. Funciona como qualquer outra substituição. Mais sobre isso no próximo apêndice.

Abaixo está mais um exemplo de pegar os resultados de uma chamada ao Claude e inserir em outra chamada mais longa. Vamos começar com o primeiro prompt (que inclui o preenchimento da resposta do Claude desta vez).

In [ ]:
first_user = """Find all names from the below text:

\"Hey, Jesse. It's me, Erin. I'm calling about the party that Joey is throwing tomorrow. Keisha said she would come and I think Mel will be there too.\""""

prefill = "<names>"

# Array de mensagens da API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": prefill
    
    }
]

# Armazena e imprime a resposta do Claude
first_response = get_completion(messages)
print("------------------------ Array completo de mensagens com substituições de variáveis ------------------------")
print(messages)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(first_response)

Vamos passar esta lista de nomes para outro prompt.

In [ ]:
second_user = "Alphabetize the list."

# Array de mensagens da API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": prefill + "\n" + first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Imprime a resposta do Claude
print("------------------------ Array completo de mensagens com substituições de variáveis ------------------------")
print(messages)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(messages))

Agora que você aprendeu sobre encadeamento de prompts, vá para o Apêndice 10.2 para aprender como implementar chamada de função usando encadeamento de prompts.

---

## Playground de Exemplos

Esta é uma área para você experimentar livremente com os exemplos de prompt mostrados nesta lição e ajustar prompts para ver como isso pode afetar as respostas do Claude.

In [ ]:
# Prompt inicial
first_user = "Name ten words that all end with the exact letters 'ab'."

# Array de mensagens da API
messages = [
    {
        "role": "user",
        "content": first_user
    }
]

# Armazena e imprime a resposta do Claude
first_response = get_completion(messages)
print(first_response)

In [ ]:
second_user = "Please find replacements for all 'words' that are not real words."

# Array de mensagens da API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Imprime a resposta do Claude
print("------------------------ Array completo de mensagens com substituições de variáveis ------------------------")
print(messages)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(messages))

In [ ]:
first_user = "Name ten words that all end with the exact letters 'ab'."

first_response = """Here are 10 words that end with the letters 'ab':

1. Cab
2. Dab
3. Grab
4. Gab
5. Jab
6. Lab
7. Nab
8. Slab
9. Tab
10. Blab"""

second_user = "Please find replacements for all 'words' that are not real words."

# Array de mensagens da API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Imprime a resposta do Claude
print("------------------------ Array completo de mensagens com substituições de variáveis ------------------------")
print(messages)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(messages))

In [ ]:
first_user = "Name ten words that all end with the exact letters 'ab'."

first_response = """Here are 10 words that end with the letters 'ab':

1. Cab
2. Dab
3. Grab
4. Gab
5. Jab
6. Lab
7. Nab
8. Slab
9. Tab
10. Blab"""

second_user = "Please find replacements for all 'words' that are not real words. If all the words are real words, return the original list."

# Array de mensagens da API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Imprime a resposta do Claude
print("------------------------ Array completo de mensagens com substituições de variáveis ------------------------")
print(messages)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(messages))

In [ ]:
# Prompt inicial
first_user = "Write a three-sentence short story about a girl who likes to run."

# Array de mensagens da API
messages = [
    {
        "role": "user",
        "content": first_user
    }
]

# Armazena e imprime a resposta do Claude
first_response = get_completion(messages)
print(first_response)

In [ ]:
second_user = "Make the story better."

# Array de mensagens da API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Imprime a resposta do Claude
print("------------------------ Array completo de mensagens com substituições de variáveis ------------------------")
print(messages)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(messages))

In [ ]:
first_user = """Find all names from the below text:

\"Hey, Jesse. It's me, Erin. I'm calling about the party that Joey is throwing tomorrow. Keisha said she would come and I think Mel will be there too.\""""

prefill = "<names>"

# Array de mensagens da API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": prefill
    
    }
]

# Armazena e imprime a resposta do Claude
first_response = get_completion(messages)
print("------------------------ Array completo de mensagens com substituições de variáveis ------------------------")
print(messages)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(first_response)

In [ ]:
second_user = "Alphabetize the list."

# Array de mensagens da API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": prefill + "\n" + first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Imprime a resposta do Claude
print("------------------------ Array completo de mensagens com substituições de variáveis ------------------------")
print(messages)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(messages))